# 📈 İnternetten Veri Çeken Döviz Takip Botu

Bu proje, Hürriyet Bigpara sitesinden canlı döviz fiyatlarını web kazıma (web scraping) yöntemiyle çekerek listeleyen ve döviz çeviri işlemleri yapan bir masaüstü uygulamasıdır.

![Uygulama Ekran Görüntüsü](./public/ekran_goruntusu.png)

Aşağıdaki hücreleri **sırasıyla (Shift + Enter)** çalıştırarak kodun nasıl adım adım inşa edildiğini inceleyebilirsiniz.

## 1. Adım: Kütüphanelerin Yüklenmesi

Uygulamamız için gerekli kütüphaneleri içe aktarıyoruz:
* `requests`: Web sitesine bağlanmak için.
* `bs4` (BeautifulSoup): HTML sayfalarını parse etmek için.
* `tkinter`: Grafik arayüzü çizmek için.

In [ ]:
import requests
from bs4 import BeautifulSoup
import tkinter as tk
from tkinter import messagebox
from tkinter import ttk

## 2. Adım: Web Scraping Fonksiyonu

Bigpara web sitesinden döviz bilgilerini (ad, son fiyat, fark, alış, satış) ayıklayan `fiyatlari_getir` fonksiyonunu tanımlıyoruz.

In [ ]:
def fiyatlari_getir():
    url = "https://bigpara.hurriyet.com.tr/doviz/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            fiyatlar = []
            symbol_infos = soup.find_all("div", class_="symbol-info")
            for info in symbol_infos:
                tr = info.find_parent("tr")
                if not tr: continue
                
                name_el = info.find(class_="symbol-name")
                doviz_cinsi = name_el.text.strip() if name_el else ""
                
                price_el = tr.find("td", {"data-column": "price"})
                fiyat = price_el.text.strip() if price_el else "0"
                
                change_el = tr.find("td", {"data-column": "change"})
                fark = change_el.text.strip() if change_el else "0"
                
                alis_el = tr.find("td", {"data-column": "volume"})
                alis = alis_el.text.strip() if alis_el else "0"
                
                satis_el = tr.find("td", {"data-column": "sales"})
                satis = satis_el.text.strip() if satis_el else "0"
                
                if doviz_cinsi:
                    fiyatlar.append({
                        "doviz": doviz_cinsi,
                        "fiyat": fiyat,
                        "fark": fark,
                        "alis": alis,
                        "satis": satis
                    })
            return fiyatlar
        return None
    except Exception:
        return None

## 3. Adım: Hesaplama Fonksiyonu

Kullanıcının girdiği miktarı banka alış ve satış kurlarını baz alarak çeviren mantıksal işlevi yazıyoruz.

In [ ]:
def hesapla():
    try:
        miktar = float(entry_miktar.get().replace(",", "."))
    except ValueError:
        messagebox.showerror("Hata", "Lütfen geçerli bir miktar girin.")
        return
    
    secilen_doviz = combobox_doviz.get()
    hedef_veri = next((d for d in doviz_listesi if d["doviz"] == secilen_doviz), None)
    if not hedef_veri: return
    
    fiyat = float(hedef_veri["fiyat"].replace(".", "").replace(",", "."))
    alis = float(hedef_veri["alis"].replace(".", "").replace(",", "."))
    satis = float(hedef_veri["satis"].replace(".", "").replace(",", "."))
    
    if alis == 0: alis = fiyat
    if satis == 0: satis = fiyat
    
    yon = var_yon.get()
    if yon == 1:
        sonuc = miktar * alis
        label_sonuc.config(text=f"{miktar:.2f} {secilen_doviz} = {sonuc:.2f} TL\n(Alış Kuru: {alis:.4f})")
    else:
        sonuc = miktar / satis
        label_sonuc.config(text=f"{miktar:.2f} TL = {sonuc:.4f} {secilen_doviz}\n(Satış Kuru: {satis:.4f})")

## 4. Adım: Arayüz ve Canvas Entegrasyonu

Tasarımı oluşturup Canvas üzerine çizim yaptığımız ve uygulamayı başlattığımız final hücresi.

In [ ]:
# Kodun tamamını tek bir hücrede birleştirip arayüzü başlatıyoruz
import requests
from bs4 import BeautifulSoup
import tkinter as tk
from tkinter import messagebox
from tkinter import ttk

def fiyatlari_getir():
    url = "https://bigpara.hurriyet.com.tr/doviz/"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            fiyatlar = []
            symbol_infos = soup.find_all("div", class_="symbol-info")
            for info in symbol_infos:
                tr = info.find_parent("tr")
                if not tr: continue
                name_el = info.find(class_="symbol-name")
                doviz_cinsi = name_el.text.strip() if name_el else ""
                price_el = tr.find("td", {"data-column": "price"})
                fiyat = price_el.text.strip() if price_el else "0"
                change_el = tr.find("td", {"data-column": "change"})
                fark = change_el.text.strip() if change_el else "0"
                alis_el = tr.find("td", {"data-column": "volume"})
                alis = alis_el.text.strip() if alis_el else "0"
                satis_el = tr.find("td", {"data-column": "sales"})
                satis = satis_el.text.strip() if satis_el else "0"
                if doviz_cinsi:
                    fiyatlar.append({"doviz": doviz_cinsi, "fiyat": fiyat, "fark": fark, "alis": alis, "satis": satis})
            return fiyatlar
        return None
    except Exception:
        return None

def hesapla():
    try:
        miktar = float(entry_miktar.get().replace(",", "."))
    except ValueError:
        messagebox.showerror("Hata", "Lütfen geçerli bir miktar girin.")
        return
    secilen_doviz = combobox_doviz.get()
    hedef_veri = next((d for d in doviz_listesi if d["doviz"] == secilen_doviz), None)
    if not hedef_veri: return
    fiyat = float(hedef_veri["fiyat"].replace(".", "").replace(",", "."))
    alis = float(hedef_veri["alis"].replace(".", "").replace(",", "."))
    satis = float(hedef_veri["satis"].replace(".", "").replace(",", "."))
    if alis == 0: alis = fiyat
    if satis == 0: satis = fiyat
    yon = var_yon.get()
    if yon == 1:
        sonuc = miktar * alis
        label_sonuc.config(text=f"{miktar:.2f} {secilen_doviz} = {sonuc:.2f} TL\n(Banka Alış Kuru: {alis:.4f})")
    else:
        sonuc = miktar / satis
        label_sonuc.config(text=f"{miktar:.2f} TL = {sonuc:.4f} {secilen_doviz}\n(Banka Satış Kuru: {satis:.4f})")

def kurları_guncelle():
    global doviz_listesi
    for item in tree.get_children():
        tree.delete(item)
    doviz_listesi = fiyatlari_getir()
    if doviz_listesi:
        for veri in doviz_listesi:
            tree.insert("", "end", values=(veri["doviz"], veri["fiyat"], veri["fark"], veri["alis"], veri["satis"]))
        combobox_doviz["values"] = [d["doviz"] for d in doviz_listesi]
        combobox_doviz.current(0)

app = tk.Tk()
app.title("Canlı Döviz Takip ve Hesaplama Botu")
app.geometry("550x700")
app.resizable(False, False)
app.configure(bg="#f1f2f6")

canvas = tk.Canvas(app, width=550, height=100, bg="#2f3542", highlightthickness=0)
canvas.pack(fill="x")
canvas.create_line(10, 85, 80, 65, 160, 75, 240, 35, 320, 50, 400, 20, 480, 30, 540, 10, fill="#2ed573", width=3, smooth=True)
canvas.create_text(275, 40, text="CANLI DÖVİZ TAKİP", fill="#ffffff", font=("Impact", 24, "bold"))
canvas.create_text(275, 75, text="Anlık veriler Bigpara sitesinden çekilmektedir.", fill="#a4b0be", font=("Arial", 10, "italic"))

tablo_frame = tk.Frame(app, bg="#f1f2f6")
tablo_frame.pack(pady=10, padx=15, fill="both", expand=True)
scrollbar = ttk.Scrollbar(tablo_frame)
scrollbar.pack(side="right", fill="y")
sutunlar = ("doviz", "fiyat", "fark", "alis", "satis")
tree = ttk.Treeview(tablo_frame, columns=sutunlar, show="headings", yscrollcommand=scrollbar.set, height=10)
tree.pack(fill="both", expand=True)
scrollbar.config(command=tree.yview)

tree.heading("doviz", text="Döviz Cinsi")
tree.heading("fiyat", text="Fiyat")
tree.heading("fark", text="Değişim (%)")
tree.heading("alis", text="Alış")
tree.heading("satis", text="Satış")

tree.column("doviz", width=120, anchor="w")
tree.column("fiyat", width=90, anchor="e")
tree.column("fark", width=90, anchor="center")
tree.column("alis", width=90, anchor="e")
tree.column("satis", width=90, anchor="e")

btn_yenile = tk.Button(app, text="Kurları Şimdi Güncelle", command=kurları_guncelle, bg="#1e90ff", fg="white", font=("Arial", 11, "bold"), bd=0, padx=15, pady=8, cursor="hand2")
btn_yenile.pack(pady=5)

hesap_paneli = tk.LabelFrame(app, text=" Döviz Çevirici / Hesap Makinesi ", bg="#f1f2f6", font=("Arial", 11, "bold"), fg="#2f3542", padx=15, pady=10)
hesap_paneli.pack(fill="x", padx=15, pady=15)

tk.Label(hesap_paneli, text="Miktar:", bg="#f1f2f6", font=("Arial", 10, "bold"), fg="#2f3542").grid(row=0, column=0, padx=5, pady=10, sticky="e")
entry_miktar = tk.Entry(hesap_paneli, width=12, font=("Arial", 11))
entry_miktar.insert(0, "100")
entry_miktar.grid(row=0, column=1, padx=5, pady=10, sticky="w")

tk.Label(hesap_paneli, text="Döviz Cinsi:", bg="#f1f2f6", font=("Arial", 10, "bold"), fg="#2f3542").grid(row=0, column=2, padx=5, pady=10, sticky="e")
combobox_doviz = ttk.Combobox(hesap_paneli, width=10, font=("Arial", 11), state="readonly")
combobox_doviz.grid(row=0, column=3, padx=5, pady=10, sticky="w")

var_yon = tk.IntVar(value=1)
rb1 = tk.Radiobutton(hesap_paneli, text="Döviz'den TL'ye (Alış)", variable=var_yon, value=1, bg="#f1f2f6", font=("Arial", 10), fg="#2f3542")
rb1.grid(row=1, column=0, columnspan=2, padx=5, pady=5, sticky="w")
rb2 = tk.Radiobutton(hesap_paneli, text="TL'den Döviz'e (Satış)", variable=var_yon, value=2, bg="#f1f2f6", font=("Arial", 10), fg="#2f3542")
rb2.grid(row=1, column=2, columnspan=2, padx=5, pady=5, sticky="w")

btn_hesapla = tk.Button(hesap_paneli, text="Dönüştür", command=hesapla, bg="#2ed573", fg="white", font=("Arial", 11, "bold"), bd=0, cursor="hand2")
btn_hesapla.grid(row=2, column=0, columnspan=4, padx=5, pady=10, sticky="we")

label_sonuc = tk.Label(hesap_paneli, text="Hesaplama sonucu burada gösterilecek.", bg="#ffffff", fg="#2f3542", font=("Arial", 11, "bold"), height=3, relief="groove", bd=1)
label_sonuc.grid(row=3, column=0, columnspan=4, padx=5, pady=10, sticky="we")

kurları_guncelle()
app.mainloop()